In [1]:
import pandas as pd

In [6]:
df=pd.read_csv('../data/sales_clean.csv',parse_dates=['InvoiceDate'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1007910 entries, 0 to 1007909
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007910 non-null  int64         
 1   StockCode    1007910 non-null  str           
 2   Description  1007910 non-null  str           
 3   Quantity     1007910 non-null  int64         
 4   InvoiceDate  1007910 non-null  datetime64[us]
 5   Price        1007910 non-null  float64       
 6   Customer ID  779423 non-null   float64       
 7   Country      1007910 non-null  str           
 8   Revenue      1007910 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(2), str(3)
memory usage: 69.2 MB


In [10]:
missing = df["Customer ID"].isna().sum()
print(f"Rows without customers: {missing} ({missing/len(df):.1%})")
print(df["InvoiceDate"].min(),df["InvoiceDate"].max())
print(df["Customer ID"].nunique())

Rows without customers: 228487 (22.7%)
2009-12-01 07:45:00 2011-12-09 12:50:00
5878


In [11]:
df=df.dropna(subset=["Customer ID"]) 
df['Customer ID']=df['Customer ID'].astype(int)   
print(df.shape,df['Customer ID'].nunique())

(779423, 9) 5878


In [15]:
cutoff=pd.Timestamp("2011-09-09")
windows_end= cutoff +pd.Timedelta(days=90)

history = df[df['InvoiceDate']<cutoff]
window = df[(df['InvoiceDate']>=cutoff) & (df['InvoiceDate']<windows_end)]

print(history['InvoiceDate'].max())
print(window['InvoiceDate'].min(), window["InvoiceDate"].max())

2011-09-08 19:58:00
2011-09-09 08:48:00 2011-12-07 17:25:00


In [17]:
bought_in_window = window['Customer ID']
order_counts = history.groupby('Customer ID')['Invoice'].nunique()
print(order_counts.head())

Customer ID
12346    11
12347     6
12348     4
12349     3
12350     1
Name: Invoice, dtype: int64


In [28]:
regular_counts = order_counts[order_counts>=5]
regular_customers = regular_counts.index
print(len(regular_customers))



1798


In [35]:
churned = (~regular_customers.isin(bought_in_window)).astype(int)
print(churned[:5],churned.sum(),len(churned))

[1 0 0 0 0] 539 1798


In [42]:
labels = pd.DataFrame({"churned":churned},index=regular_customers)
labels.head()


,churned
Customer ID,
12346,1
12347,0
12352,0
12356,0
12359,0


In [43]:
print(labels.shape, labels["churned"].mean())

(1798, 1) 0.29977753058954393
